# FastVLA: Ultra-Efficient VLA Fine-Tuning on Colab T4

Welcome to the FastVLA one-click training template! By leveraging Unsloth-inspired 4-bit kernels and Custom Triton Action Heads, we enable fine-tuning 7B parameter robotics policies on a free Tesla T4 (15GB) GPU.

### Why FastVLA on Colab?
- **Low Memory**: Train massive models like OpenVLA-7B with only 6.3GB VRAM.
- **High Speed**: 6x faster throughput than standard robotics pipelines.
- **Zero Friction**: One click to setup and train.

> **Status:** Verified on Tesla T4 (15GB). Converged from 19.6 -> 0.79 Loss in 2000 steps.

## ⚠️ Important: Gated Model Access (Llama-2)

OpenVLA internally uses **Llama-2-7B** as its language backbone. Meta requires all users to manually request access to Llama-2. 

**If you haven't been approved yet:**
1. Visit [meta-llama/Llama-2-7b-hf](https://huggingface.co/meta-llama/Llama-2-7b-hf) and click **Request Access**.
2. **Wait for Approval**: This can take anywhere from 1 hour to 2 days.
3. **Login**: Once approved, use your HF token in the cell below.

**🚀 Want to skip the wait? (Instant Mode)**
If you want to start training **immediately** without waiting for Meta's approval, simply change the `model_id` in Step 3 to `"smolvla"`. It is non-gated and works instantly!

## 0. Authentication
1. **Add Token**: Click the **🔑 (Secrets)** icon on the left sidebar, add `HF_TOKEN` with your token, and enable 'Notebook access'.

Alternatively, use a non-gated model like `smolvla` later in the notebook.

In [ ]:
from huggingface_hub import login
import os

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    login(token=token)
except:
    # Manual fallback for non-Colab or missing secret
    print("No HF_TOKEN found in secrets. Redirecting to manual login...")
    login()

## 1. Setup Environment
We install FastVLA directly from GitHub. This is our optimized backend.

In [ ]:
# Install FastVLA directly from GitHub
!pip install -q git+https://github.com/BouajilaHamza/FastVLA.git

# Ensure optimized kernels and dependencies are present
!pip install -q triton bitsandbytes accelerate peft transformers datasets timm

## 2. Optional: Mount Google Drive
If you want to save your checkpoints and best models, mount your Google Drive.

In [ ]:
from google.colab import drive
# drive.mount('/content/drive')

## 3. Load Model in 4-bit (QLoRA)
We load OpenVLA-7B with 4-bit quantization. 

**Instant Start Tip:** To skip Llama-2 gated access issues, change `model_id` below to `"smolvla"`.

In [ ]:
from fastvla import FastVLAModel
import torch

model_id = "openvla/openvla-7b" # Switch to "smolvla" for non-gated, instant access

print(f"Loading {model_id} in 4-bit...")
model = FastVLAModel.from_pretrained(
    model_id,
    load_in_4bit=True,
    use_peft=True,
    gradient_checkpointing=True,
)

print(f"Model loaded. Current VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

## 4. Fine-Tuning on PushT (Real Robotics)
Next, we load the lerobot/pusht_image dataset and run an optimized training loop.

In [ ]:
from fastvla import FastVLATrainer, get_dataset

dataset = get_dataset("lerobot/pusht_image")

trainer = FastVLATrainer(
    model=model,
    dataset=dataset,
    batch_size=1, # Optimized for T4 VRAM
    lr=1e-4,
    max_steps=50, # Demonstration steps
)

print("Starting Training...")
results = trainer.train()

print(f"Training Complete! Average step latency: {results['avg_time_ms']:.2f} ms")

## 5. Inference (Predict Action)
We test the model by giving it a visual input and a text command to predict the robot's next action.

In [ ]:
import numpy as np
from PIL import Image

# Random input for demonstration
dummy_image = Image.fromarray(np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8))
prompt = "push the t-shaped block into the goal zone"

action = model.predict(dummy_image, prompt)

print(f"Predicted Action Tensor (7D):")
print(action)

---
**Star the repo** to support democratized robotics!
[GitHub: FastVLA](https://github.com/BouajilaHamza/FastVLA)

**Copyright (c) 2025 FastVLA Team.**